# Sliver Data Transformation 

In [0]:
df = spark.read.format("delta")\
                .option("header",True)\
                  .option("inferSchema",True)\
                    .load("abfss://bronze@netflixprojectdlanju.dfs.core.windows.net/netflix_titles")

In [0]:
df.display()

# TransFormation

**_Remove the nulls_**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = df.fillna({"duration_minutes": 0, "duration_seasons": 1})

In [0]:
df.display()

**_Change the datatype of the cols._**

In [0]:
df = df.withColumn("duration_minutes",col("duration_minutes").cast(IntegerType()))\
        .withColumn("duration_seasons",col("duration_seasons").cast(IntegerType()))

In [0]:
df.printSchema()

In [0]:
df.display()

**_get the titles before the colon (:)_**

In [0]:
df = df.withColumn("short_title",split("title", ":")[0])
df.display()

**_get the value of before dash (-) of rating col._**

In [0]:
df = df.withColumn("rating",split("rating","-")[0])
df.display()

**_create a flag depends on the type_**

In [0]:
df = df.withColumn("type_flag",when(col("type")=="Movie",1).when(col("type")=="TV Show",2).otherwise(0))
df.display()

**_Rank your data_**

In [0]:
from pyspark.sql.window import *

In [0]:
df = df.withColumn("duration_ranking",dense_rank().over(Window.orderBy(col("duration_minutes").desc())))
df.display()

**USING MYSQL - TRANSFORMATIONS**

In [0]:
df.createOrReplaceTempView("temp_view")

In [0]:
df = spark.sql("""
               select * from temp_view
               """)

In [0]:
df.display()

**Aggregate data**

In [0]:
df_visual = df.groupBy("type").agg(count("*").alias("Total_count"))
display(df_visual)

In [0]:
df.write.format("delta").mode("overwrite").option("path","abfss://sliver@netflixprojectdlanju.dfs.core.windows.net/netflix_titles").save()